# Experiment 1: instrumental interference

**Question.** How much does each kind of non-vocal stem interfere with extracting the vocal?

**Design.** For every MUSDB18 test song the vocal is added to one non-vocal stem: `bass`, `drums`, `other`, or `accompaniment` (which is drums + bass + other, not a single instrument). Spleeter and Demucs (2-stem mode) then separate the vocal, and the estimate is scored against the true vocal stem.

This notebook only calls the scripts in `src/`. Every step skips finished work. Set the paths in `src/config.py` (or the `SSS_DATA_ROOT` and `MUSDB18_ROOT` environment variables) first.

In [ ]:
import os, subprocess, sys
from pathlib import Path

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO / "src"))
from config import EXP1_DEMUCS_DIR, EXP1_PAIRS_DIR, EXP1_RESULTS_DIR, EXP1_SPLEETER_DIR

EXP = REPO / "src"

# Interpreters for the three environments (see requirements/). Edit the last two.
PY_DATA = sys.executable                                   # this notebook's kernel: data-and-evaluation env
PY_SPLEETER = "/path/to/spleeter_env/bin/python"
PY_DEMUCS = "/path/to/demucs_env/bin/python"

def run(cmd, cwd=REPO, env=None):
    """Run a command and stream its output into the notebook."""
    with subprocess.Popen(cmd, cwd=cwd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as p:
        for line in p.stdout:
            print(line, end="")

## 1. Build the paired mixtures (about 5 minutes)

In [ ]:
run([PY_DATA, str(EXP / "make_pairs_exp1.py")])

## 2. Separate with Spleeter, 2-stem (about 10 minutes)

In [ ]:
run([PY_SPLEETER, "-u", str(EXP / "run_spleeter_exp1.py"), str(EXP1_PAIRS_DIR), str(EXP1_SPLEETER_DIR)])

## 3. Separate with Demucs, 2-stem
The shell script runs from the folder that holds the song folders, and needs the `demucs` command on the PATH.

In [ ]:
env = dict(os.environ, PATH=str(Path(PY_DEMUCS).parent) + os.pathsep + os.environ["PATH"])
run(["bash", str(EXP / "run_demucs_exp1.sh")], cwd=EXP1_PAIRS_DIR, env=env)

## 4. Score
Vocal SI-SDR, an SI-SAR-like artifact ratio, and RMSE against the true vocal stem, on mono downmixes. The summary is the **median** over the 50 songs; this is what reproduces the numbers in the original write-up. (SI-SAR-like and RMSE are non-standard, and RMSE depends on level, so define them if you report them.)

In [ ]:
run([PY_DATA, str(EXP / "evaluate_exp1.py"), "--workers", "6"])

## 5. Results

In [ ]:
import pandas as pd
from IPython.display import Image, display

summary = pd.read_csv(EXP1_RESULTS_DIR / "summary_median.csv")
display(summary.round(3))
display(Image(str(EXP1_RESULTS_DIR / "fig_exp1_median_si_sdr.png")))

## Reading the result
- Demucs beats Spleeter on every pairing.
- Both models recover the vocal best from bass and drums and worst from `other` and accompaniment.
- Why (harmonic overlap with the vocal) is a hypothesis; it has not been tested here.